In [0]:
def read_raw_roads():
    return spark.readStream.table('traffic_data.bronze.raw_roads')

In [0]:
%run "./00.Common"

In [0]:
def road_category_name(df):
    from pyspark.sql import functions as F
    df_clean = (df.withColumn( 'Road_Category_Name' , 
                              F.when(F.col('Road_category')== 'TA','Class A Truch Road')
                                .when(F.col('Road_category')== 'TM','Class A Truch Motor')
                                .when(F.col('Road_category')== 'PA','Class A Principle Road')
                                .when(F.col('Road_category')== 'PM','Class A Principal Motorway')
                                .when(F.col('Road_category')== 'M', 'Class B road')
                                .otherwise('NA')
                            )   
                )
    print('Road Category Name is added')
    return df_clean

In [0]:
def road_type(df):
    from pyspark.sql import functions as F
    df_clean = (
        df.withColumn('Road_Type', 
                        F.when(F.col('Road_Category_Name').like('Class A%'), 'Major')
                         .when(F.col('Road_Category_Name').like('Class B%'), 'Minor')
                        .otherwise('NA')
                        )
    )
    print('Road Type is added')
    return df_clean

In [0]:
def write_silver_raw_roads(df):
    source = spark.sql('DESCRIBE EXTERNAL LOCATION trafficprojectexternallocation').select('url').collect()[0][0]
    ws = (
        df.writeStream
          .format('delta')
          .option('checkpointLocation', f'{source}/Checkpoint/Silver/RoadsLoad')
          .outputMode('append')
          .queryName('RoadsLoad')
          .trigger(availableNow = True)
          .toTable('traffic_data.silver.roadsload')
    )
    ws.awaitTermination()
    print('Silver Road Load is completed')

In [0]:
df1 = read_raw_roads()
df2 = remove_dup(df1)
df3 = replace_null(df2)
df4 = road_category_name(df3)
df5 = road_type(df4)
write_silver_raw_roads(df5)

In [0]:
%sql
select * from traffic_data.silver.roadsload